# Eval Trained Adapter v3 — Diagnostic on Held-Out Slice (Full Corpus)

Loads the LoRA adapter trained in `gideonnazarian/nemotron-lora-sft-v3` (from the dataset `gideonnazarian/nemotron-lora-sft-v3-adapter`) onto Nemotron 3 Nano and runs the same 5-problem held-out eval as v6/v7.

**Why:** v3 scored 0.58 on the leaderboard (+0.04 vs v2 0.54). v3 trained on the full 4681-record corpus (6× v2's 750 records) with identical Fix A reasoning. The v7 diagnostic confirmed v2 had the right mechanism but suffered from arithmetic precision and mapping derivation slips. We expect v3 to:
- Reduce small arithmetic errors (more examples = better numeric muscle memory)
- Improve cipher mapping derivation (605 vs 150 cipher examples)
- Keep correct on problems v7 already nailed (1, 3)

**Same holdout as v5/v6/v7:** last 5 of `train.csv` by id sort — `ffce9e31`, `ffd5bada`, `ffd89354`, `ffdfb678`, `ffe680c2`.

**Reference:** v5 untrained 0/5 strict, 1/5 substring. v6 v1 adapter 1/5 strict, 1/5 substring. v7 v2 adapter 2/5 strict, 3/5 substring.

In [ ]:
# Cell 1: Setup (sitedir + ptxas) — same boilerplate as other kernels
import os, random, shutil, site, stat, sys, time, glob
import numpy as np
import torch

T0 = time.perf_counter()
def step(label):
    print(f"[{time.perf_counter() - T0:7.1f}s] {label}", flush=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

for utility_root in ("/kaggle/input/notebooks", "/kaggle/usr/lib/notebooks", "/kaggle/lib"):
    if not os.path.isdir(utility_root):
        continue
    for root, dirs, _ in os.walk(utility_root):
        if "models" in root.split(os.sep):
            dirs[:] = []
            continue
        if os.path.basename(root) == "python_packages":
            site.addsitedir(root)

PTXAS_DST = "/tmp/ptxas-blackwell"
for src in (
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell",
    "/kaggle/input/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell",
):
    if os.path.exists(src):
        if not os.path.exists(PTXAS_DST):
            shutil.copy(src, PTXAS_DST)
            os.chmod(PTXAS_DST, stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
        os.environ["TRITON_PTXAS_PATH"] = PTXAS_DST
        os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = PTXAS_DST
        break
step("Setup done")

In [ ]:
# Cell 2: Load base model + tokenizer + ATTACH trained adapter via PeftModel
import kagglehub
import mamba_ssm  # noqa: F401
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
step(f"Tokenizer loaded; IM_END_ID={IM_END_ID}")

# Find the trained adapter dir — v3 adapter (Fix A reasoning, full corpus, LB 0.58)
ADAPTER_CANDIDATES = [
    "/kaggle/input/datasets/gideonnazarian/nemotron-lora-sft-v3-adapter",
    "/kaggle/input/nemotron-lora-sft-v3-adapter",
]
ADAPTER_PATH = None
for p in ADAPTER_CANDIDATES:
    if os.path.isdir(p) and os.path.exists(os.path.join(p, "adapter_config.json")):
        ADAPTER_PATH = p
        break
if ADAPTER_PATH is None:
    hits = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    if hits:
        ADAPTER_PATH = os.path.dirname(hits[0])
if ADAPTER_PATH is None:
    raise FileNotFoundError(f"adapter_config.json not found. /kaggle/input/ contains: {os.listdir('/kaggle/input')}")
step(f"ADAPTER_PATH={ADAPTER_PATH}")

t = time.perf_counter()
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map="auto", trust_remote_code=True, dtype=torch.bfloat16,
)
step(f"Base model loaded in {time.perf_counter() - t:.0f}s")

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
step(f"Trained adapter attached via PeftModel.from_pretrained")
step(f"VRAM={torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# Cell 3: Run inference on the same 5 held-out problems as v5
import re
import json
import pandas as pd

PROMPT_SUFFIX = (
    "\nPlease put your final answer inside `\\boxed{}`. "
    "For example: `\\boxed{your answer}`"
)
HOLDOUT_N = 5
MAX_NEW_TOKENS = 1024

TRAIN_CSV = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
df = pd.read_csv(TRAIN_CSV).sort_values("id").reset_index(drop=True)
holdout = df.tail(HOLDOUT_N).reset_index(drop=True)
step(f"Held-out: {list(holdout['id'])}")

BOXED_RE = re.compile(r"\\boxed\{([^}]*)\}")
THINK_CLOSE = "</think>"
IM_END = "<|im_end|>"

def extract_after_think(c):
    if THINK_CLOSE not in c: return ""
    post = c.split(THINK_CLOSE, 1)[1]
    if IM_END in post: post = post.split(IM_END, 1)[0]
    return post.strip()

def extract_boxed_anywhere(c):
    matches = list(BOXED_RE.finditer(c))
    return matches[-1].group(1).strip() if matches else ""

def gold_substring(c, g):
    return g if (g and g in c) else ""

results = []
for i, row in holdout.iterrows():
    pid = row["id"]; gold = str(row["answer"])
    prompt_text = row["prompt"] + PROMPT_SUFFIX
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt_text}],
        return_tensors="pt", add_generation_prompt=True, return_dict=True,
    ).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    t_p = time.perf_counter()
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
            eos_token_id=[IM_END_ID, tokenizer.eos_token_id],
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.perf_counter() - t_p
    n_new = out.shape[1] - prompt_len
    completion = tokenizer.decode(out[0, prompt_len:], skip_special_tokens=False)
    after = extract_after_think(completion)
    boxed = extract_boxed_anywhere(completion)
    substr = gold_substring(completion, gold)
    results.append({
        "id": pid, "completion_tokens": n_new, "seconds": elapsed,
        "after_think": after, "boxed_anywhere": boxed, "gold": gold,
        "completion": completion,
        "correct_after_think": after.strip() == gold.strip(),
        "correct_boxed": boxed.strip() == gold.strip(),
        "correct_substring": bool(substr),
        "has_think_close": THINK_CLOSE in completion,
    })
    print(
        f"  [{i+1}/{HOLDOUT_N}] id={pid} | {n_new:4d} tok in {elapsed:5.1f}s | "
        f"close={THINK_CLOSE in completion} | boxed={boxed!r:25s} | gold={gold!r:15s} | "
        f"strict={'✓' if boxed.strip()==gold.strip() else '✗'} substr={'✓' if substr else '✗'}",
        flush=True,
    )

In [ ]:
# Cell 4: Summary + save full completions for offline inspection
n = len(results)
summary = {
    "n": n,
    "version": "v8-trained-adapter-v3",
    "adapter": "gideonnazarian/nemotron-lora-sft-v3 (Fix A, full 4681 corpus, LB=0.58)",
    "think_closed_rate": sum(r["has_think_close"] for r in results) / n,
    "acc_after_think": sum(r["correct_after_think"] for r in results) / n,
    "acc_boxed": sum(r["correct_boxed"] for r in results) / n,
    "acc_substring": sum(r["correct_substring"] for r in results) / n,
    "avg_tokens": sum(r["completion_tokens"] for r in results) / n,
}
print("\n=== Summary (v3 trained adapter, Fix A + full corpus) ===")
print(json.dumps(summary, indent=2))

print("\n=== Comparison across all eval runs ===")
print("  v5 untrained baseline:    0/5 strict, 1/5 substring (gravity only)")
print("  v6 trained adapter v1:    1/5 strict, 1/5 substring (unit_conv only)")
print("  v7 trained adapter v2:    2/5 strict, 3/5 substring (bit_manip, cipher#3, gravity-math)")
print(f"  v8 trained adapter v3:    {sum(r['correct_boxed'] for r in results)}/{n} strict, "
      f"{sum(r['correct_substring'] for r in results)}/{n} substring")

# Save
pd.DataFrame(results).to_csv("/kaggle/working/eval_results.csv", index=False)
with open("/kaggle/working/eval_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

In [ ]:
# Cell 5: Last 600 chars per problem for offline analysis
for i, r in enumerate(results, 1):
    print(f"=== Problem {i} (id={r['id']}, gold={r['gold']!r}) ===")
    print(f"close={r['has_think_close']} | boxed={r['boxed_anywhere']!r} | substr={r['correct_substring']}")
    print("--- LAST 600 CHARS ---")
    print(r["completion"][-600:])
    print()
step(f"Total wall time: {(time.perf_counter() - T0) / 60:.1f} min")